# 第10課 - 生產環境中的 AI 代理

在本課中您將學習使用 Microsoft Agent Framework (Python) 的 AI 代理的**生產模式**。我們涵蓋：

- **可觀測性 (Observability)** — 在代理互動中加入計時與日誌記錄
- **評估 (Evaluation)** — 使用評估代理為回應質量打分
- **成本管理 (Cost management)** — 代幣優化與模型選擇的策略

情境是一個**旅遊代理**，幫助用戶規劃行程，並在其上層加入監控與評估。


## 設定


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## 生產環境考量

將 AI 代理從原型移轉到生產環境，需要特別注意三大支柱：

1. **可觀察性** — 您需要能看見代理在做什麼、花了多長時間，以及它呼叫了哪些工具。沒有追蹤與記錄，除錯生產環境問題幾乎不可能。

2. **評估** — 自動化品質檢查可確保代理的回應隨時間保持準確、完整且有幫助。評估代理可以根據已定義的標準對回應進行評分。

3. **成本管理** — Token 使用量直接影響成本。像提示優化、模型選擇和快取等策略可以在不犧牲品質的情況下幫助控制開支。


## 建立可觀測的代理

我們定義旅行工具，並用計時包裝代理呼叫，以便監測延遲。在生產環境中，你會整合 OpenTelemetry 或類似的追蹤後端。


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## 評估模式

一個常見的生產模式是使用第二個代理作為 **評估者**。評估者會根據預先定義的標準（例如完整性、準確性和有用性）來評分主要代理的回應。

這可實現：
- 在回應到達使用者之前的自動化品質把關
- 在提示或模型變更時的回歸偵測
- 持續監測代理的效能表現


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## 成本管理策略

控制成本對於生產環境的 AI 代理至關重要。以下是主要策略：

| 策略 | 說明 |
|---|---|
| **提示優化** | 保持系統指令簡潔。移除多餘的上下文以減少輸入 tokens。 |
| **模型選擇** | 對於分類或抽取等簡單任務，使用較小、較便宜的模型（例如 GPT-4o-mini），將較大的模型保留給複雜推理任務。 |
| **快取** | 快取工具結果和頻繁查詢以避免重複的 API 呼叫。 |
| **Token 預算** | 設定 `max_tokens` 限制以防止回應意外過長。 |
| **批次處理** | 在可能的情況下，將多個使用者查詢合併為單一的 API 呼叫。 |

在實務上，分層的方法效果良好：將簡單的請求導向快速且低成本的模型，僅將複雜查詢升級至更有能力的模型。


## 摘要

在本課中，您學到了如何：

1. **加入可觀察性** 到代理互動中，使用計時與日誌記錄，為追蹤與監控奠定基礎。
2. **評估代理回應** 自動使用評估代理，對完整性、準確性和有用性進行評分。
3. **管理成本**：透過提示優化、模型選擇、快取與 token 預算來控制成本。

這些生產實務模式有助於確保您的 AI 代理在大規模運作時可靠、可衡量並具成本效益。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免責聲明：
本文件經由人工智慧翻譯服務 Co-op Translator (https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們力求準確，但自動翻譯可能包含錯誤或不準確之處，敬請留意。原始文件的原文應視為具有權威性的版本。對於重要或關鍵資訊，建議採用專業人工翻譯。我們對因使用本翻譯所產生的任何誤解或誤讀概不負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
